In [2]:
# Extract English words from ConceptNet CSV with specific relationships
import csv
import json
from tqdm import tqdm
import re
import spacy
import numpy as np
import pandas as pd
nlp = spacy.load("en_core_web_sm")

In [3]:
PARSE_CSV = False
UNWEIGHT_JSON = False
MATRIX = True

In [4]:
# Relationships we care about
# grab from the full CSV file downloaded from https://github.com/commonsense/conceptnet5/wiki/Downloads

if PARSE_CSV:
    target_rels = {"/r/UsedFor", "/r/CapableOf", "/r/ReceivesAction"}

    graph = {}

    CSV_FILE = "/Users/mcharit2/Downloads/full_conceptnet_ass.csv"

    # First, count total lines for tqdm progress bar
    with open(CSV_FILE, encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)

    with open(CSV_FILE, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter='\t')
        for row in tqdm(reader, total=total_lines, desc="Processing ConceptNet"):
            if len(row) < 4:
                continue

            rel, uri1, uri2 = row[1], row[2], row[3]  # col0=ID, col1=rel, col2=start, col3=end

            # Only keep target relationships
            if rel not in target_rels:
                continue

            # Only keep if both ends are English
            if uri1.startswith("/c/en/") and uri2.startswith("/c/en/"):
                
                # get the subject
                subject = uri1.split("/")[3].replace("_", " ")
                object = uri2.split("/")[3].replace("_", " ")
                if subject not in graph:
                    graph[subject] = []
                graph[subject].append(object)

    # Save to file
    with open("../bank_files/conceptnet_graph_raw.json", "w", encoding="utf-8") as f:
        json.dump(graph, f, ensure_ascii=False, indent=2)


In [5]:
def get_verb_noun(phrase):
    ''' Extract the main verb and noun from a phrase using spaCy '''
    doc_obj = nlp(phrase)
    verb = None
    noun = None
    for token in doc_obj:
        if token.pos_ == "NOUN":
            noun = token.lemma_
            
        elif token.pos_ == "VERB":
            verb = token.lemma_
        
        if verb and noun:
            break
    return verb, noun

In [6]:
# parse the raw graph into a more structured format based on conceptnet_exp into noweight format
# exported as a JSON

if UNWEIGHT_JSON:
    graph = json.load(open("../bank_files/conceptnet_graph_raw.json", "r", encoding="utf-8"))
    word_graph = {}
    for subj, obj_list in tqdm(graph.items(), desc="Parsing graph into word graph"):
        if " " in subj:
            continue  # skip multi-word subjects
        if subj not in word_graph:
            word_graph[subj] = {}
        for obj in obj_list:
            # use spacy to determine pos in obj
            verb, noun = get_verb_noun(obj)

            if verb and noun:
                if verb not in word_graph[subj]:
                    word_graph[subj][verb] = []
                if noun not in word_graph[subj][verb]:
                    word_graph[subj][verb].append(noun)

    # save to file
    with open("bank_files/full_word_graph_noweight.json", "w", encoding="utf-8") as f:
        json.dump(word_graph, f, ensure_ascii=False, indent=2)


In [ ]:
# create a 2D matrix of directed graph relationships between nouns with the entries being a list of verbs
# row: subject noun
# col: object noun

# index: verb 

if MATRIX:
    graph = json.load(open("../bank_files/conceptnet_graph_raw.json", "r", encoding="utf-8"))
    word_graph = {}
    

    subject_list = []
    object_list = []
    verb_index = {}

    # lemmatize the subject and objects and grab the verb and noun from the object phrase
    for s, phrases in tqdm(graph.items(), desc="Parsing graph into word graph"):
        subj = nlp(s.lower())[0].lemma_

        if " " in subj or len(subj) < 3 or not re.match("^[a-zA-Z]+$", subj):
            continue  # skip multi-word subjects

        if subj not in word_graph:          # initialize subject in word graph if not already there
            word_graph[subj] = {}

        if subj not in subject_list:        # add to subject list if not already there
            subject_list.append(subj)

        # break down phrases into verb and noun
        for obj in phrases:
            # use spacy to determine pos in obj
            verb, noun = get_verb_noun(obj)

            if verb and noun:
                if verb not in verb_index:
                    verb_index[verb] = len(verb_index)

                if noun not in word_graph[subj]:
                    word_graph[subj][noun] = []
                if verb not in word_graph[subj][noun]:
                    word_graph[subj][noun].append(verb_index[verb])

                
                if noun not in object_list:
                    object_list.append(noun)

    print("Creating matrix with {} subjects, {} objects, and {} verbs".format(len(subject_list), len(object_list), len(verb_index))))

    # create the matrix (pandas dataframe might be easier)
    matrix = pd.DataFrame(index=subject_list, columns=object_list)

    for subj in subject_list:
        for obj in object_list:
            if obj in word_graph.get(subj, {}):
                matrix.loc[subj, obj] = word_graph[subj][obj]

    # set NaN to empty list
    matrix = matrix.applymap(lambda x: x if isinstance(x, list) else [])


    print("Exporting to files...")

    # save matrix to file as a pandas dataframe
    matrix.to_csv("../bank_files/cn_full_matrix.csv")

    # save keys to numpy files
    np.save("../bank_files/cn_full_subjects.npy", np.array(subject_list))
    np.save("../bank_files/cn_full_objects.npy", np.array(object_list))
    np.save("../bank_files/cn_full_verbs.npy", np.array(list(verb_index.keys())))



Parsing graph into word graph: 100%|██████████| 13852/13852 [06:54<00:00, 33.40it/s]
/var/folders/n4/ztbcwq6n25947kmq8sjw2v180000gp/T/ipykernel_53227/1449467279.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  matrix = matrix.applymap(lambda x: x if isinstance(x, list) else [])


In [ ]:
# view dataframe
matrix.head()

,map,leprosy,series,dust,insect,fetus,pregnancy,vote,wife,professor,...,gardener,combat,standing,ego,liveliness,condolence,know,fancy,literatur,debug
aaa,[0],[],[],[],[],[],[],[],[],[],...,[],[],[],[],[],[],[],[],[],[]
aardvark,[],[],[],[],[],[],[],[],[],[],...,[],[],[],[],[],[],[],[],[],[]
aarmadillo,[],[0],[],[],[],[],[],[],[],[],...,[],[],[],[],[],[],[],[],[],[]
abacus,[],[],[1],[],[],[],[],[],[],[],...,[],[],[],[],[],[],[],[],[],[]
abandon,[],[],[],[2],[],[],[],[],[],[],...,[],[],[],[],[],[],[],[],[],[]


In [20]:
verb_list = np.load("../bank_files/cn_full_verbs.npy", allow_pickle=True)
print(matrix.shape)

(6467, 4856)
